#  Импорты и базовые настройки

In [ ]:
# # ==============================================================================
# # ЯЧЕЙКА 1: Импорты и настройка устройства
# # ==============================================================================
# import os
# import random
# import numpy as np
# import cv2
# import tifffile as tiff
# from tqdm.auto import tqdm

# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader-+
# import torchvision.transforms.functional as TF

# # Оптимизация выделения памяти для A6000
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"✅ Устройство: {device}")
# if torch.cuda.is_available():
#     print(f"✅ Видеокарта: {torch.cuda.get_device_name(0)}")
#     print(f"✅ Доступно VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# # Глобальные константы нормализации для ваших TIFF (16-bit)
# GLOBAL_MAX = 65535.0

Unknown instance spec: Please select VM configuration

# Архитектура (Guided Restormer + Noise Estimator)

In [ ]:
# # ==============================================================================
# # ЯЧЕЙКА 2: Архитектура Guided Restormer
# # ==============================================================================

# class SigmaEstimator(nn.Module):
#     """Маленькая подсеть для предсказания уровня шума (Sigma) по шумному патчу."""
#     def __init__(self):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(inplace=True),
#             nn.MaxPool2d(2),
#             nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True),
#             nn.MaxPool2d(2),
#             nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True),
#             nn.AdaptiveAvgPool2d(1), # Схлопывает всю картинку в 1 пиксель
#             nn.Flatten(),
#             nn.Linear(64, 16), nn.ReLU(inplace=True),
#             nn.Linear(16, 1),
#             nn.Softplus() # Гарантирует, что предсказанная сигма всегда > 0
#         )

#     def forward(self, x):
#         return self.net(x)

# # --- Блоки Restormer ---
# class MDTA(nn.Module):
#     """Multi-Dconv Head Transposed Attention (Внимание по каналам)"""
#     def __init__(self, channels, num_heads):
#         super().__init__()
#         self.num_heads = num_heads
#         self.temperature = nn.Parameter(torch.ones(1, num_heads, 1, 1))
#         self.qkv = nn.Conv2d(channels, channels * 3, kernel_size=1, bias=False)
#         self.qkv_dwconv = nn.Conv2d(channels * 3, channels * 3, kernel_size=3, padding=1, groups=channels * 3, bias=False)
#         self.project_out = nn.Conv2d(channels, channels, kernel_size=1, bias=False)

#     def forward(self, x):
#         b, c, h, w = x.shape
#         qkv = self.qkv_dwconv(self.qkv(x))
#         q, k, v = qkv.chunk(3, dim=1)
        
#         q = q.reshape(b, self.num_heads, -1, h * w)
#         k = k.reshape(b, self.num_heads, -1, h * w)
#         v = v.reshape(b, self.num_heads, -1, h * w)
        
#         q, k = F.normalize(q, dim=-1), F.normalize(k, dim=-1)
#         attn = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * self.temperature, dim=-1)
#         out = self.project_out(torch.matmul(attn, v).reshape(b, c, h, w))
#         return out

# class GDFN(nn.Module):
#     """Gated-Dconv Feed-Forward Network"""
#     def __init__(self, channels, expansion_factor=2.66):
#         super().__init__()
#         hidden_channels = int(channels * expansion_factor)
#         self.project_in = nn.Conv2d(channels, hidden_channels * 2, kernel_size=1, bias=False)
#         self.dwconv = nn.Conv2d(hidden_channels * 2, hidden_channels * 2, kernel_size=3, padding=1, groups=hidden_channels * 2, bias=False)
#         self.project_out = nn.Conv2d(hidden_channels, channels, kernel_size=1, bias=False)

#     def forward(self, x):
#         x1, x2 = self.dwconv(self.project_in(x)).chunk(2, dim=1)
#         return self.project_out(F.gelu(x1) * x2)

# class TransformerBlock(nn.Module):
#     def __init__(self, channels, num_heads):
#         super().__init__()
#         self.norm1 = nn.LayerNorm(channels)
#         self.attn = MDTA(channels, num_heads)
#         self.norm2 = nn.LayerNorm(channels)
#         self.ffn = GDFN(channels)

#     def forward(self, x):
#         b, c, h, w = x.shape
#         # LayerNorm требует формат [B, H, W, C]
#         x_norm = self.norm1(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
#         x = x + self.attn(x_norm)
#         x_norm = self.norm2(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
#         x = x + self.ffn(x_norm)
#         return x

# class GuidedRestormer(nn.Module):
#     """Главная сеть: Оценивает шум + Чистит картинку"""
#     def __init__(self, dim=48, num_blocks=4, num_heads=4):
#         super().__init__()
#         self.sigma_estimator = SigmaEstimator()
        
#         # Restormer принимает 3 канала: [Noisy, Canny, SigmaMap]
#         self.embed = nn.Conv2d(3, dim, kernel_size=3, padding=1)
#         self.blocks = nn.Sequential(*[TransformerBlock(dim, num_heads) for _ in range(num_blocks)])
#         self.mapping = nn.Conv2d(dim, 1, kernel_size=3, padding=1)

#     def forward(self, noisy, canny, true_sigma=None):
#         # 1. Предсказываем сигму (даже если есть true_sigma, чтобы обучать Estimator)
#         pred_sigma = self.sigma_estimator(noisy)
        
#         # 2. Формируем карту шума. 
#         # При обучении (если передана true_sigma) Restormer смотрит на идеальную сигму (Teacher Forcing).
#         # На инференсе использует предсказанную.
#         sigma_val = true_sigma if (self.training and true_sigma is not None) else pred_sigma
        
#         B, _, H, W = noisy.shape
#         sigma_map = sigma_val.view(B, 1, 1, 1).expand(B, 1, H, W)
        
#         # 3. Склеиваем 3 канала
#         x = torch.cat([noisy, canny, sigma_map], dim=1)
        
#         # 4. Прогоняем через Restormer
#         fea = self.embed(x)
#         fea = self.blocks(fea)
#         predicted_noise = self.mapping(fea)
        
#         # 5. Global Residual (Вычитаем шум из оригинала)
#         cleaned = noisy - predicted_noise
#         cleaned = torch.clamp(cleaned, 0.0, 1.0)
        
#         return cleaned, pred_sigma

# print("✅ Архитектура GuidedRestormer загружена!")

Unknown instance spec: Please select VM configuration

# Оптимизированный Data Pipeline 

In [ ]:
# # ==============================================================================
# # ЯЧЕЙКА 3: Оптимизированная подготовка данных (Shift Logic & Canny)
# # ==============================================================================

# def get_canny_full_image(img_numpy):
#     """Применяет Canny к полной картинке (с предварительным размытием)."""
#     # Перевод в 8-bit для Canny
#     img_8u = np.clip((img_numpy / GLOBAL_MAX) * 255, 0, 255).astype(np.uint8)
#     blurred = cv2.GaussianBlur(img_8u, (5, 5), 0)
#     edges = cv2.Canny(blurred, 65, 140)
#     return (edges / 255.0).astype(np.float32)

# def extract_patches(img, patch_size, stride):
#     """Нарезает 2D массив на патчи."""
#     h, w = img.shape
#     patches = []
#     for y in range(0, h - patch_size + 1, stride):
#         for x in range(0, w - patch_size + 1, stride):
#             patches.append(img[y:y+patch_size, x:x+patch_size])
#     return patches

# def build_epoch_data(epoch, all_pairs, step=5, patch_size=512, overlap=0.2):
#     """ 
#     Магия оптимизации: 
#     1. Выбирает картинки со сдвигом (0, 5, 10... потом 1, 6, 11...).
#     2. Открывает картинку РОВНО 1 РАЗ.
#     3. Делает Canny, режет на патчи, считает Sigma и отдает в RAM.
#     """
#     # Сдвиг эпохи (0, 1, 2, 3, 4, 0, 1...)
#     start_offset = epoch % step
#     epoch_pairs = all_pairs[start_offset::step]
    
#     stride = int(patch_size * (1.0 - overlap))
    
#     data_noisy, data_clean, data_canny, data_sigma = [], [], [], []
    
#     print(f"📦 Подготовка данных для Эпохи {epoch+1} (Сдвиг: {start_offset}). Чтение {len(epoch_pairs)} снимков...")
    
#     for noisy_path, clean_path in tqdm(epoch_pairs, leave=False):
#         # 1. ЧИТАЕМ РОВНО 1 РАЗ
#         n_img = tiff.imread(noisy_path).astype(np.float32)
#         c_img = tiff.imread(clean_path).astype(np.float32)
        
#         # 2. CANNY НА ВСЮ КАРТИНКУ
#         canny_img = get_canny_full_image(n_img)
        
#         # 3. НАРЕЗКА НА ПАТЧИ
#         n_patches = extract_patches(n_img, patch_size, stride)
#         c_patches = extract_patches(c_img, patch_size, stride)
#         canny_patches = extract_patches(canny_img, patch_size, stride)
        
#         # 4. ОБРАБОТКА ПАТЧЕЙ
#         for n_p, c_p, can_p in zip(n_patches, c_patches, canny_patches):
#             # Нормализация
#             n_p_norm = np.clip(n_p / GLOBAL_MAX, 0.0, 1.0)
#             c_p_norm = np.clip(c_p / GLOBAL_MAX, 0.0, 1.0)
            
#             # Считаем TRUE SIGMA для патча (Noisy - Clean)
#             sigma = np.std(n_p_norm - c_p_norm)
            
#             data_noisy.append(n_p_norm)
#             data_clean.append(c_p_norm)
#             data_canny.append(can_p)
#             data_sigma.append(sigma)
            
#     return np.array(data_noisy), np.array(data_clean), np.array(data_canny), np.array(data_sigma, dtype=np.float32)

Unknown instance spec: Please select VM configuration

# Аугментация и Dataset

In [ ]:
# # ==============================================================================
# # ЯЧЕЙКА 4: Аугментация и PyTorch Dataset
# # ==============================================================================

# def d4_augmentation_no_crop(noisy_t, clean_t, canny_t):
#     """D4 Аугментация (отражения + повороты). Применяется ко всем 3 тензорам синхронно!"""
#     if random.random() > 0.5:
#         noisy_t = TF.hflip(noisy_t)
#         clean_t = TF.hflip(clean_t)
#         canny_t = TF.hflip(canny_t)
#     if random.random() > 0.5:
#         noisy_t = TF.vflip(noisy_t)
#         clean_t = TF.vflip(clean_t)
#         canny_t = TF.vflip(canny_t)
        
#     k = random.randint(0, 3)
#     if k > 0:
#         noisy_t = torch.rot90(noisy_t, k, dims=[1, 2])
#         clean_t = torch.rot90(clean_t, k, dims=[1, 2])
#         canny_t = torch.rot90(canny_t, k, dims=[1, 2])
        
#     return noisy_t, clean_t, canny_t

# class EpochRAMDataset(Dataset):
#     """Датасет, который хранит уже нарезанные патчи для текущей эпохи в RAM."""
#     def __init__(self, n_arr, c_arr, canny_arr, sigmas, apply_aug=True):
#         self.n_arr = n_arr
#         self.c_arr = c_arr
#         self.canny_arr = canny_arr
#         self.sigmas = sigmas
#         self.apply_aug = apply_aug

#     def __len__(self):
#         return len(self.n_arr)

#     def __getitem__(self, idx):
#         # Берем из RAM и переводим в тензоры [1, H, W]
#         n_t = torch.from_numpy(self.n_arr[idx]).unsqueeze(0)
#         c_t = torch.from_numpy(self.c_arr[idx]).unsqueeze(0)
#         canny_t = torch.from_numpy(self.canny_arr[idx]).unsqueeze(0)
#         sigma_val = torch.tensor(self.sigmas[idx])
        
#         if self.apply_aug:
#             n_t, c_t, canny_t = d4_augmentation_no_crop(n_t, c_t, canny_t)
            
#         return n_t, c_t, canny_t, sigma_val

Unknown instance spec: Please select VM configuration

# Инициализация функций потерь (Loss Functions)

In [ ]:
# # ==============================================================================
# # ЯЧЕЙКА 5: Инициализация функций потерь (Loss Functions)
# # ==============================================================================
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# class CharbonnierLoss(nn.Module):
#     def __init__(self, eps=1e-3):
#         super().__init__()
#         self.eps2 = eps ** 2

#     def forward(self, pred, target):
#         return torch.mean(torch.sqrt((pred - target)**2 + self.eps2))


# class EdgeLoss(nn.Module):
#     """Градиенты Собеля по X и Y (штраф за смазанные края)"""
#     def __init__(self):
#         super().__init__()
#         sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
#         sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
#         self.register_buffer('sobel_x', sobel_x)
#         self.register_buffer('sobel_y', sobel_y)

#     def forward(self, pred, target):
#         pred_x = F.conv2d(pred, self.sobel_x.to(pred.device), padding=1)
#         pred_y = F.conv2d(pred, self.sobel_y.to(pred.device), padding=1)
#         target_x = F.conv2d(target, self.sobel_x.to(pred.device), padding=1)
#         target_y = F.conv2d(target, self.sobel_y.to(pred.device), padding=1)
#         return F.l1_loss(pred_x, target_x) + F.l1_loss(pred_y, target_y)


# class FullHybridCTLoss(nn.Module):
#     """
#     Полный гибридный лосс: Пространство (Charbonnier) + Градиенты (Edge)
#     """
#     def __init__(self, w_charb=0.5, w_edge=0.5):
#         super().__init__()
#         self.w_charb = w_charb
#         self.w_edge = w_edge

#         self.charb = CharbonnierLoss()
#         self.edge = EdgeLoss()

#     def forward(self, pred, target):
#         l_charb = self.charb(pred, target)
#         l_edge = self.edge(pred, target)

#         return (self.w_charb * l_charb) + (self.w_edge * l_edge)

# print("✅ Функции потерь (FullHybridCTLoss) успешно загружены!")

Unknown instance spec: Please select VM configuration

# Главный цикл обучения

In [ ]:
# # ==============================================================================
# # ЯЧЕЙКА 6: Главный цикл обучения (Training Loop)
# # ==============================================================================

# # --- 1. НАСТРОЙКИ ОБУЧЕНИЯ ---
# NUM_EPOCHS = 50
# STEP_SHIFT = 5      # Шаг сдвига (каждую 5-ю картинку)
# BATCH_SIZE = 16     # Для A6000 и патчей 512x512 можно увеличить до 24-32, если хватит VRAM

# # Инициализация модели
# model = GuidedRestormer().to(device)

# # Инициализация лоссов
# criterion_img = FullHybridCTLoss(w_charb=0.5, w_edge=0.5) # Твой лосс для картинки
# criterion_sigma = nn.L1Loss()                             # Лосс для подсети предсказания сигмы (1 число)

# # Оптимизатор и Scaler для смешанной точности (ускоряет обучение на A6000)
# optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
# scaler = torch.cuda.amp.GradScaler() 

# # ВНИМАНИЕ: Замени train_files_list на твою реальную переменную со списком файлов для обучения
# # Например, ту, что генерировалась при разбиении на Train/Val/Test
# all_train_pairs = train_files_list # <--- УБЕДИСЬ, ЧТО ПЕРЕМЕННАЯ НАЗЫВАЕТСЯ ПРАВИЛЬНО

# print(f"🚀 ЗАПУСК ДИНАМИЧЕСКОГО ОБУЧЕНИЯ НА {NUM_EPOCHS} ЭПОХ...")

# for epoch in range(NUM_EPOCHS):
#     # 1. ГЕНЕРИРУЕМ ДАННЫЕ ДЛЯ ТЕКУЩЕЙ ЭПОХИ
#     # Функция откроет нужные картинки 1 раз, применит Canny ко всему снимку и нарежет патчи в RAM
#     n_arr, c_arr, canny_arr, sigmas = build_epoch_data(
#         epoch=epoch, 
#         all_pairs=all_train_pairs, 
#         step=STEP_SHIFT, 
#         patch_size=512
#     )
    
#     # Оборачиваем массивы из RAM в Dataset
#     epoch_dataset = EpochRAMDataset(n_arr, c_arr, canny_arr, sigmas, apply_aug=True)
    
#     # Создаем DataLoader (shuffle=True перемешает патчи со всех загруженных снимков)
#     train_loader = DataLoader(epoch_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    
#     model.train()
#     running_loss_img = 0.0
#     running_loss_sigma = 0.0
    
#     pbar = tqdm(train_loader, desc=f"Эпоха {epoch+1}/{NUM_EPOCHS}")
#     for n_b, c_b, canny_b, sigma_b in pbar:
#         # Перенос на GPU
#         n_b = n_b.to(device, non_blocking=True)
#         c_b = c_b.to(device, non_blocking=True)
#         canny_b = canny_b.to(device, non_blocking=True)
#         sigma_b = sigma_b.to(device, non_blocking=True).unsqueeze(1) # Делаем размер [Batch, 1]
        
#         optimizer.zero_grad(set_to_none=True)
        
#         # Автоматическая смешанная точность (AMP)
#         with torch.cuda.amp.autocast():
#             # Прогон модели. Передаем true_sigma, чтобы обучать SigmaEstimator
#             pred_clean, pred_sigma = model(n_b, canny_b, true_sigma=sigma_b)
            
#             # Считаем лоссы
#             loss_img = criterion_img(pred_clean, c_b)         # FullHybridCTLoss
#             loss_sigma = criterion_sigma(pred_sigma, sigma_b) # L1Loss для скаляра
            
#             # Общий лосс. Вес 0.1 для сигмы нужен, чтобы сеть фокусировалась на картинке,
#             # а предсказание сигмы шло как вспомогательная задача.
#             total_loss = loss_img + 0.1 * loss_sigma
            
#         # Обратный проход
#         scaler.scale(total_loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
        
#         # Статистика
#         running_loss_img += loss_img.item()
#         running_loss_sigma += loss_sigma.item()
        
#         pbar.set_postfix(L_img=f"{loss_img.item():.4f}", L_sig=f"{loss_sigma.item():.4f}")
        
#     avg_img_loss = running_loss_img / len(train_loader)
#     print(f"🏁 Эпоха {epoch+1} завершена. Avg Hybrid Loss: {avg_img_loss:.4f}")
    
#     # ========================================================================
#     # ОЧИСТКА ПАМЯТИ ПЕРЕД СЛЕДУЮЩЕЙ ЭПОХОЙ (Критически важно!)
#     # ========================================================================
#     del n_arr, c_arr, canny_arr, sigmas, epoch_dataset, train_loader
#     torch.cuda.empty_cache()

#     # Сохранение чекпоинта
#     checkpoint_path = f"restormer_guided_epoch_{epoch+1}.pth"
#     torch.save(model.state_dict(), checkpoint_path)

Unknown instance spec: Please select VM configuration

# Импорт библиотек и конфигурация под DataSphere

In [19]:
# ==============================================================================
# 1. ИМПОРТ БИБЛИОТЕК И НАСТРОЙКА DATASPHERE
# ==============================================================================
import os
# Безопасная настройка под PyTorch 2.0.1 (без крашей ядра)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import gc
import re
import csv
import json
import time
import random
import numpy as np
import cv2
import tifffile as tiff
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

# Очистка памяти перед стартом
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ DataSphere GPU активен: {gpu_name} (Всего VRAM: {vram_gb:.1f} ГБ)")
else:
    device = torch.device("cpu")
    print("⚙️ Вычисления на CPU")

print("✅ Все библиотеки успешно импортированы!")

✅ DataSphere GPU активен: Tesla V100-PCIE-32GB (Всего VRAM: 31.7 ГБ)
✅ Все библиотеки успешно импортированы!


 # МОДУЛЬ 1: Чистая функция нарезки (Slicer)

In [20]:
# ==============================================================================
# МОДУЛЬ 1: Чистая функция нарезки (Slicer)
# ==============================================================================
def slice_image_to_patches(image_or_path, patch_size=512, overlap_ratio=0.2):
    """
    ОТВЕТСТВЕННОСТЬ: Только геометрическая нарезка снимка.
    Возвращает:
      - patches: numpy массив (N, patch_size, patch_size)
      - coords: список координат (y, x)
      - orig_shape: размеры (H, W)
    """
    if isinstance(image_or_path, str):
        img = tiff.imread(image_or_path).astype(np.float32)
    else:
        img = image_or_path.astype(np.float32)

    h, w = img.shape
    stride = int(patch_size * (1.0 - overlap_ratio))

    y_steps = list(range(0, h - patch_size + 1, stride))
    if y_steps[-1] != h - patch_size: y_steps.append(h - patch_size)

    x_steps = list(range(0, w - patch_size + 1, stride))
    if x_steps[-1] != w - patch_size: x_steps.append(w - patch_size)

    patches, coords = [], []
    for y in y_steps:
        for x in x_steps:
            patches.append(img[y:y+patch_size, x:x+patch_size])
            coords.append((y, x))

    return np.array(patches), coords, (h, w)

print("✅ Модуль 1 (Нарезчик Slicer) готов!")

✅ Модуль 1 (Нарезчик Slicer) готов!


# МОДУЛЬ 2: Чистая D4-аугментация (Без кропа)

In [21]:
# ==============================================================================
# МОДУЛЬ 2: Аугментация D4 + Параметр Гамма-коррекции для (Noisy, Clean, Canny)
# ==============================================================================
def d4_gamma_canny_augmentation(noisy_tensor, clean_tensor, canny_tensor, gamma_range=(0.9, 1.1)):
    """
    Синхронная аугментация для 3-х карт:
    1. Гамма-коррекция (случайное изменение контраста I^gamma)
    2. Отражения по горизонтали и вертикали (Flips)
    3. Повороты на 90, 180, 270 градусов
    """
    # 1. 🔥 ПАРАМЕТР ГАММА
    if gamma_range is not None:
        gamma = random.uniform(gamma_range[0], gamma_range[1])
        noisy_tensor = torch.clamp(noisy_tensor ** gamma, 0.0, 1.0)
        clean_tensor = torch.clamp(clean_tensor ** gamma, 0.0, 1.0)

    # 2. Отражения (50% шанс)
    if random.random() > 0.5:
        noisy_tensor = TF.hflip(noisy_tensor)
        clean_tensor = TF.hflip(clean_tensor)
        canny_tensor = TF.hflip(canny_tensor)
        
    if random.random() > 0.5:
        noisy_tensor = TF.vflip(noisy_tensor)
        clean_tensor = TF.vflip(clean_tensor)
        canny_tensor = TF.vflip(canny_tensor)

    # 3. Повороты кратно 90°
    k = random.randint(0, 3)
    if k > 0:
        noisy_tensor = torch.rot90(noisy_tensor, k, dims=[1, 2])
        clean_tensor = torch.rot90(clean_tensor, k, dims=[1, 2])
        canny_tensor = torch.rot90(canny_tensor, k, dims=[1, 2])

    return noisy_tensor, clean_tensor, canny_tensor

print("✅ Модуль 2 (Аугментация с Гаммой и Canny) готов!")

✅ Модуль 2 (Аугментация с Гаммой и Canny) готов!


# МОДУЛЬ 3: Буферный датасет под DataSphere (CTBufferedDataset)

In [22]:
# ==============================================================================
# МОДУЛЬ 3: Ультра-легкий буферный датасет (Генерирует Noisy + Canny на лету)
# ==============================================================================
class CTBufferedDataset(Dataset):
    """
    Буферный датасет:
    - Считывает снимок 3067x3067 ровно 1 раз
    - Генерирует карту Canny в компактном uint8
    - Формирует 2-канальный вход [Noisy, Canny]
    - Расход памяти: всего ~150-200 МБ
    """
    def __init__(
        self, 
        noisy_dir, 
        clean_dir, 
        file_list=None, 
        patch_size=512, 
        overlap_ratio=0.2, 
        chunk_size=8,
        transform_fn=None
    ):
        self.patch_size = patch_size
        self.chunk_size = chunk_size
        self.transform_fn = transform_fn

        # 1. Список файлов
        if file_list is not None:
            self.common_names = file_list
        else:
            noisy_files = sorted([f for f in os.listdir(noisy_dir) if f.lower().endswith(('.tif', '.tiff'))])
            clean_files = sorted([f for f in os.listdir(clean_dir) if f.lower().endswith(('.tif', '.tiff'))])
            self.common_names = sorted(list(set(noisy_files) & set(clean_files)))

        self.noisy_paths = [os.path.join(noisy_dir, f) for f in self.common_names]
        self.clean_paths = [os.path.join(clean_dir, f) for f in self.common_names]

        # 2. Базовые координаты сетки
        _, self.base_coords, _ = slice_image_to_patches(self.noisy_paths[0], patch_size, overlap_ratio)

        self._buffer_cache = {}
        self.patch_index = []
        self.reshuffle()

        print(f"📁 Загружено пар снимков: {len(self.common_names)} (от {self.common_names[0]} до {self.common_names[-1]})")
        print(f"🎉 Сформирован буферный индекс: {len(self.patch_index)} патчей (Буфер = {chunk_size} снимков ~ 150 МБ RAM)")
        print(f"🔄 Стратегия аугментации: {transform_fn.__name__ if transform_fn else 'Без аугментации'}\n")

    def reshuffle(self):
        """Перемешивает снимки и жестко очищает кэш перед новой эпохой."""
        self._buffer_cache.clear()
        gc.collect()

        self.patch_index = []
        img_indices = list(range(len(self.common_names)))
        random.shuffle(img_indices)

        for i in range(0, len(img_indices), self.chunk_size):
            chunk_imgs = img_indices[i:i + self.chunk_size]
            chunk_patches = []
            for img_idx in chunk_imgs:
                for y, x in self.base_coords:
                    chunk_patches.append((img_idx, y, x))
            
            random.shuffle(chunk_patches)
            self.patch_index.extend(chunk_patches)

    def __len__(self):
        return len(self.patch_index)

    def _load_image_pair_to_buffer(self, img_idx):
        """Считывает снимок 1 раз и генерирует Canny в uint8."""
        n_raw = tiff.imread(self.noisy_paths[img_idx]).astype(np.float32)
        c_raw = tiff.imread(self.clean_paths[img_idx]).astype(np.float32)

        # Нормализация в компактный uint8 [0..255] (экономит 75% RAM)
        n_u8 = ((n_raw - np.min(n_raw)) / (np.max(n_raw) - np.min(n_raw) + 1e-8) * 255.0).astype(np.uint8)
        c_u8 = ((c_raw - np.min(c_raw)) / (np.max(c_raw) - np.min(c_raw) + 1e-8) * 255.0).astype(np.uint8)

        # 🔥 Генерируем карту Canny
        blurred = cv2.GaussianBlur(n_u8, (3, 3), 1.0)
        canny_u8 = cv2.Canny(blurred, 65, 140)

        self._buffer_cache[img_idx] = (n_u8, c_u8, canny_u8)

    def __getitem__(self, idx):
        img_idx, y, x = self.patch_index[idx]

        if img_idx not in self._buffer_cache:
            if len(self._buffer_cache) >= self.chunk_size:
                oldest_key = next(iter(self._buffer_cache))
                del self._buffer_cache[oldest_key]

            self._load_image_pair_to_buffer(img_idx)

        n_u8, c_u8, canny_u8 = self._buffer_cache[img_idx]
        p = self.patch_size

        n_patch = n_u8[y:y+p, x:x+p]
        c_patch = c_u8[y:y+p, x:x+p]
        canny_patch = canny_u8[y:y+p, x:x+p]

        # Перевод в тензоры [1, H, W]
        n_tensor = torch.from_numpy(n_patch).unsqueeze(0).float() / 255.0
        c_tensor = torch.from_numpy(c_patch).unsqueeze(0).float() / 255.0
        canny_tensor = torch.from_numpy(canny_patch).unsqueeze(0).float() / 255.0

        # Применяем аугментацию (D4 + Гамма)
        if self.transform_fn is not None:
            n_tensor, c_tensor, canny_tensor = self.transform_fn(n_tensor, c_tensor, canny_tensor)

        # 🔥 Склеиваем Noisy (1 канал) + Canny (1 канал) в 2-канальный вход [2, 512, 512]
        input_2ch = torch.cat([n_tensor, canny_tensor], dim=0)

        return input_2ch, c_tensor

print("✅ Модуль 3 (Датасет с 2-канальным входом Noisy + Canny) готов!")

✅ Модуль 3 (Датасет с 2-канальным входом Noisy + Canny) готов!


# Инициализация и Разделение 80 / 10 / 10

In [23]:
# ==============================================================================
# ⚙️ СТАБИЛЬНЫЕ ПАРАМЕТРЫ ДЛЯ RESTORMER (VRAM: ~6.5 ГБ из 32 ГБ)
# ==============================================================================
PATCH_SIZE = 256 # 🔥 256x256 снижает расход VRAM в 4 раза!
CHUNK_SIZE = 8   # 8 снимков в буфере RAM (~150 МБ)
STEP = 5         # Каждый 5-й снимок
BATCH_SIZE = 4   # 🔥 Батч 4 гарантирует 100% стабильность без OOM

NOISY_DIR = "filestore/filestorage/V_beton30_angle05"
CLEAN_DIR = "filestore/filestorage/V_beton30_angle005"
# ==============================================================================

def get_file_number(filename):
    nums = re.findall(r'\d+', filename)
    return int(nums[-1]) if nums else -1

# 1. Поиск всех пар файлов
noisy_files = sorted([f for f in os.listdir(NOISY_DIR) if f.lower().endswith(('.tif', '.tiff'))])
clean_files = sorted([f for f in os.listdir(CLEAN_DIR) if f.lower().endswith(('.tif', '.tiff'))])
all_pairs = sorted(list(set(noisy_files) & set(clean_files)))

# 2. Фильтрация краевых срезов [0..400] и [2430..2816] + шаг STEP=5
clean_range_pairs = [
    f for f in all_pairs 
    if not (0 <= get_file_number(f) <= 400 or 2430 <= get_file_number(f) <= 2816)
]
subsampled_pairs = clean_range_pairs[::STEP]

# 3. Воспроизводимое разделение 80 / 10 / 10 (seed=42)
random.seed(42)
shuffled_pairs = subsampled_pairs.copy()
random.shuffle(shuffled_pairs)

n_total = len(shuffled_pairs)          # 406 снимков
n_train = int(0.80 * n_total)          # 324 снимка (80%)
n_val   = int(0.10 * n_total)          # 40 снимков (10%)
n_test  = n_total - n_train - n_val    # 42 снимка (10%)

train_files = sorted(shuffled_pairs[:n_train])
val_files   = sorted(shuffled_pairs[n_train:n_train + n_val])
test_files  = sorted(shuffled_pairs[n_train + n_val:])

print("=" * 75)
print(f"📊 ПАРАМЕТРЫ RESTORMER (Патчи {PATCH_SIZE}x{PATCH_SIZE}, Батч {BATCH_SIZE}):")
print(f"   🟢 Train (80%): {len(train_files)} пар — D4 + Гамма")
print(f"   🟡 Val   (10%): {len(val_files)} пар — для чекпоинтов")
print(f"   🔴 Test  (10%): {len(test_files)} пар — НЕПРИКОСНОВЕННЫЙ ТЕСТ")
print("=" * 75 + "\n")

# 4. Создаем датасеты на 256x256
train_dataset = CTBufferedDataset(
    noisy_dir=NOISY_DIR,
    clean_dir=CLEAN_DIR,
    file_list=train_files,
    patch_size=PATCH_SIZE,                # 🔥 256x256
    overlap_ratio=0.2,
    chunk_size=CHUNK_SIZE,
    transform_fn=d4_gamma_canny_augmentation
)

val_dataset = CTBufferedDataset(
    noisy_dir=NOISY_DIR,
    clean_dir=CLEAN_DIR,
    file_list=val_files,
    patch_size=PATCH_SIZE,                # 🔥 256x256
    overlap_ratio=0.2,
    chunk_size=CHUNK_SIZE,
    transform_fn=None
)

test_dataset = CTBufferedDataset(
    noisy_dir=NOISY_DIR,
    clean_dir=CLEAN_DIR,
    file_list=test_files,
    patch_size=PATCH_SIZE,                # 🔥 256x256
    overlap_ratio=0.2,
    chunk_size=CHUNK_SIZE,
    transform_fn=None
)

# 5. DataLoader'ы
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"🎉 ИТОГ: Train={len(train_dataset)} | Val={len(val_dataset)} | Test={len(test_dataset)} (VRAM защищена!)")

📊 ПАРАМЕТРЫ RESTORMER (Патчи 256x256, Батч 4):
   🟢 Train (80%): 324 пар — D4 + Гамма
   🟡 Val   (10%): 40 пар — для чекпоинтов
   🔴 Test  (10%): 42 пар — НЕПРИКОСНОВЕННЫЙ ТЕСТ

📁 Загружено пар снимков: 324 (от rec_00401.tif до rec_02426.tif)
🎉 Сформирован буферный индекс: 72900 патчей (Буфер = 8 снимков ~ 150 МБ RAM)
🔄 Стратегия аугментации: d4_gamma_canny_augmentation

📁 Загружено пар снимков: 40 (от rec_00511.tif до rec_02386.tif)
🎉 Сформирован буферный индекс: 9000 патчей (Буфер = 8 снимков ~ 150 МБ RAM)
🔄 Стратегия аугментации: Без аугментации

📁 Загружено пар снимков: 42 (от rec_00416.tif до rec_02411.tif)
🎉 Сформирован буферный индекс: 9450 патчей (Буфер = 8 снимков ~ 150 МБ RAM)
🔄 Стратегия аугментации: Без аугментации

🎉 ИТОГ: Train=72900 | Val=9000 | Test=9450 (VRAM защищена!)


# МОДУЛЬ 4: Архитектура Restormer (Zamir et al., 2022)

In [24]:
# ==============================================================================
# МОДУЛЬ 4: Стабильный 2-канальный Guided Restormer для PyTorch 2.0.1
# ==============================================================================
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

class LayerNorm2d(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.bias = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=1, keepdim=True)
        var = x.var(dim=1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(var + self.eps) * self.weight + self.bias


class MDTA(nn.Module):
    """Multi-Dconv Head Transposed Attention (Канальное внимание)"""
    def __init__(self, channels, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(1, num_heads, 1, 1))
        self.qkv = nn.Conv2d(channels, channels * 3, kernel_size=1, bias=False)
        self.qkv_dwconv = nn.Conv2d(channels * 3, channels * 3, kernel_size=3, padding=1, groups=channels * 3, bias=False)
        self.project_out = nn.Conv2d(channels, channels, kernel_size=1, bias=False)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv_dwconv(self.qkv(x))
        q, k, v = qkv.chunk(3, dim=1)

        q = q.reshape(b, self.num_heads, -1, h * w)
        k = k.reshape(b, self.num_heads, -1, h * w)
        v = v.reshape(b, self.num_heads, -1, h * w)

        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        attn = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * self.temperature, dim=-1)
        out = torch.matmul(attn, v).reshape(b, c, h, w)
        return self.project_out(out)


class GDFN(nn.Module):
    """Gated-Dconv Feed-Forward Network"""
    def __init__(self, channels, expansion_factor=2.66):
        super().__init__()
        hidden_channels = int(channels * expansion_factor)
        self.project_in = nn.Conv2d(channels, hidden_channels * 2, kernel_size=1, bias=False)
        self.dwconv = nn.Conv2d(hidden_channels * 2, hidden_channels * 2, kernel_size=3, padding=1, groups=hidden_channels * 2, bias=False)
        self.project_out = nn.Conv2d(hidden_channels, channels, kernel_size=1, bias=False)

    def forward(self, x):
        x1, x2 = self.dwconv(self.project_in(x)).chunk(2, dim=1)
        return self.project_out(F.gelu(x1) * x2)


class TransformerBlock(nn.Module):
    def __init__(self, channels, num_heads):
        super().__init__()
        self.norm1 = LayerNorm2d(channels)
        self.attn = MDTA(channels, num_heads)
        self.norm2 = LayerNorm2d(channels)
        self.ffn = GDFN(channels)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class Downsample(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, in_channels * 2, kernel_size=2, stride=2, bias=False)
    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2, bias=False)
    def forward(self, x):
        return self.conv(x)


class RestormerCT(nn.Module):
    """
    Guided Restormer (in_channels=2: Noisy + Canny, out_channels=1: Clean)
    """
    def __init__(self, in_channels=2, out_channels=1, dim=48, num_blocks=[2, 3, 3, 4], num_heads=[1, 2, 4, 8]):
        super().__init__()
        self.in_conv = nn.Conv2d(in_channels, dim, kernel_size=3, padding=1, bias=False)

        # Энкодер
        self.encoder1 = nn.Sequential(*[TransformerBlock(dim, num_heads[0]) for _ in range(num_blocks[0])])
        self.down1 = Downsample(dim)

        self.encoder2 = nn.Sequential(*[TransformerBlock(dim*2, num_heads[1]) for _ in range(num_blocks[1])])
        self.down2 = Downsample(dim*2)

        self.encoder3 = nn.Sequential(*[TransformerBlock(dim*4, num_heads[2]) for _ in range(num_blocks[2])])
        self.down3 = Downsample(dim*4)

        # Горлышко
        self.bottleneck = nn.Sequential(*[TransformerBlock(dim*8, num_heads[3]) for _ in range(num_blocks[3])])

        # Декодер
        self.up3 = Upsample(dim*8)
        self.reduce3 = nn.Conv2d(dim*8, dim*4, kernel_size=1, bias=False)
        self.decoder3 = nn.Sequential(*[TransformerBlock(dim*4, num_heads[2]) for _ in range(num_blocks[2])])

        self.up2 = Upsample(dim*4)
        self.reduce2 = nn.Conv2d(dim*4, dim*2, kernel_size=1, bias=False)
        self.decoder2 = nn.Sequential(*[TransformerBlock(dim*2, num_heads[1]) for _ in range(num_blocks[1])])

        self.up1 = Upsample(dim*2)
        self.reduce1 = nn.Conv2d(dim*2, dim, kernel_size=1, bias=False)
        self.decoder1 = nn.Sequential(*[TransformerBlock(dim, num_heads[0]) for _ in range(num_blocks[0])])

        # Выходной слой
        self.out_conv = nn.Conv2d(dim, out_channels, kernel_size=3, padding=1, bias=False)

    def forward(self, x):
        orig_noisy = x[:, 0:1, :, :] # Выделяем канал Noisy
        feat = self.in_conv(x)

        f1 = self.encoder1(feat)
        f2 = self.encoder2(self.down1(f1))
        f3 = self.encoder3(self.down2(f2))

        b = self.bottleneck(self.down3(f3))

        d3 = self.decoder3(self.reduce3(torch.cat([self.up3(b), f3], dim=1)))
        d2 = self.decoder2(self.reduce2(torch.cat([self.up2(d3), f2], dim=1)))
        d1 = self.decoder1(self.reduce1(torch.cat([self.up1(d2), f1], dim=1)))

        # Global Residual: Чистый = Шумный МИНУС Предсказанный шум
        predicted_noise = self.out_conv(d1)
        cleaned = orig_noisy - predicted_noise
        return cleaned

# Инициализация модели
model = RestormerCT(in_channels=2, out_channels=1, dim=48).to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🚀 Guided Restormer успешно создан на {device}! Параметров: {num_params:,}")

🚀 Guided Restormer успешно создан на cuda! Параметров: 11,627,892


# МОДУЛЬ 5: Функция потерь FullHybridCTLoss

In [25]:
# ==============================================================================
# МОДУЛЬ 5: Функция потерь без создания мусорных тензоров на GPU
# ==============================================================================
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps2 = eps ** 2

    def forward(self, pred, target):
        return torch.mean(torch.sqrt((pred - target)**2 + self.eps2))


class EdgeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def forward(self, pred, target):
        pred_x = F.conv2d(pred, self.sobel_x, padding=1)
        pred_y = F.conv2d(pred, self.sobel_y, padding=1)
        target_x = F.conv2d(target, self.sobel_x, padding=1)
        target_y = F.conv2d(target, self.sobel_y, padding=1)
        return F.l1_loss(pred_x, target_x) + F.l1_loss(pred_y, target_y)


class FullHybridCTLoss(nn.Module):
    def __init__(self, w_charb=1.0, w_edge=0.15):
        super().__init__()
        self.w_charb = w_charb
        self.w_edge = w_edge
        self.charb = CharbonnierLoss(eps=1e-3)
        self.edge = EdgeLoss()

    def forward(self, pred, target):
        return (self.w_charb * self.charb(pred, target)) + (self.w_edge * self.edge(pred, target))

criterion = FullHybridCTLoss(w_charb=1.0, w_edge=0.15).to(device)
print("✅ Функция потерь FullHybridCTLoss готова!")

✅ Функция потерь FullHybridCTLoss готова!


# МОДУЛЬ 6: Цикл обучения (Training Loop) с логированием

In [26]:
# ==============================================================================
# МОДУЛЬ 6: Цикл обучения с контролем чистой VRAM
# ==============================================================================
import gc
import csv
import json
import time
import torch.optim as optim
from tqdm.auto import tqdm

# 1. Проверяем, что видеокарта чистая
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    allocated_gb = torch.cuda.memory_allocated() / (1024 ** 3)
    free_gb = (torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)) - allocated_gb
    print(f"📊 Статус VRAM перед стартом: Занято = {allocated_gb:.2f} ГБ | СВОБОДНО = {free_gb:.2f} ГБ")
    assert free_gb > 10.0, "⚠️ Память видеокарты не очищена! Перезапустите Kernel -> Restart Kernel."

optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

NUM_EPOCHS = 30
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

use_amp = (device.type == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

best_val_loss = float('inf')
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "restormer_best_model.pth")
LAST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "restormer_last_checkpoint.pth")
CSV_LOG_PATH = os.path.join(CHECKPOINT_DIR, "loss_history.csv")
JSON_LOG_PATH = os.path.join(CHECKPOINT_DIR, "loss_history.json")

# Инициализируем CSV
with open(CSV_LOG_PATH, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss", "lr", "epoch_time_sec"])

loss_history = {"epoch": [], "train_loss": [], "val_loss": [], "lr": []}

print("=" * 80)
print(f"🚀 ЗАПУСК ОБУЧЕНИЯ GUIDED RESTORMER НА {NUM_EPOCHS} ЭПОХ (Патчи 256x256, Батч {BATCH_SIZE})")
print(f"⚡ Аппаратное ускорение (AMP FP16): {'ВКЛЮЧЕНО' if use_amp else 'Выключено'}")
print("=" * 80 + "\n")

for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    # Перемешивание буфера
    train_dataset.reshuffle()

    # 1. TRAIN
    model.train()
    train_running_loss = 0.0
    train_loop = tqdm(train_loader, desc=f"Эпоха [{epoch+1:02d}/{NUM_EPOCHS:02d}] TRAIN", unit="батч")

    for noisy_2ch_b, clean_b in train_loop:
        noisy_2ch_b = noisy_2ch_b.to(device, non_blocking=True)
        clean_b = clean_b.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            preds = model(noisy_2ch_b)
            loss = criterion(preds, clean_b)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_val = loss.item()
        train_running_loss += loss_val
        train_loop.set_postfix(train_loss=f"{loss_val:.4f}", lr=f"{optimizer.param_groups[0]['lr']:.6f}")

    train_loss = train_running_loss / len(train_loader)

    # 2. VAL
    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for noisy_2ch_b, clean_b in val_loader:
            noisy_2ch_b = noisy_2ch_b.to(device, non_blocking=True)
            clean_b = clean_b.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                preds = model(noisy_2ch_b)
                loss = criterion(preds, clean_b)

            val_running_loss += loss.item()

    val_loss = val_running_loss / len(val_loader)
    epoch_time = time.time() - start_time
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    # Логирование
    loss_history["epoch"].append(epoch + 1)
    loss_history["train_loss"].append(train_loss)
    loss_history["val_loss"].append(val_loss)
    loss_history["lr"].append(current_lr)

    with open(CSV_LOG_PATH, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch + 1, train_loss, val_loss, current_lr, epoch_time])

    with open(JSON_LOG_PATH, 'w') as f:
        json.dump(loss_history, f, indent=4)

    print(f"⏱️ Эпоха {epoch+1:02d} ({epoch_time:.1f}с) | Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f} | LR: {current_lr:.6f}")

    # Сохранение лучшей модели
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"   ⭐ [НОВЫЙ РЕКОРД НА VAL] Val Loss = {best_val_loss:.5f} -> Сохранено в '{BEST_MODEL_PATH}'")

    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'best_val_loss': best_val_loss
    }
    torch.save(checkpoint, LAST_CHECKPOINT_PATH)

    # Принудительный сброс памяти
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print("\n" + "=" * 80)
print(f"🎉 ОБУЧЕНИЕ ЗАВЕРШЕНО! Лучший Val Loss: {best_val_loss:.5f}")
print("=" * 80)

Эпоха [01/30] TRAIN: 100%|██████████| 18225/18225 [1:53:06<00:00,  2.69батч/s, lr=0.000200, train_loss=0.0954]


⏱️ Эпоха 01 (7078.1с) | Train Loss: 0.09916 | Val Loss: 0.09736 | LR: 0.000200
   ⭐ [НОВЫЙ РЕКОРД НА VAL] Val Loss = 0.09736 -> Сохранено в 'checkpoints/restormer_best_model.pth'


Эпоха [02/30] TRAIN:   0%|          | 33/18225 [00:15<2:18:25,  2.19батч/s, lr=0.000199, train_loss=0.0726]


KeyboardInterrupt: 

# МОДУЛЬ 7: Расчет метрик PSNR/SSIM на Тестовой выборке (TEST SET)

In [ ]:
# ==============================================================================
# ВИЗУАЛИЗАЦИЯ КРИВЫХ ОБУЧЕНИЯ (LOSS CURVES)
# ==============================================================================
if os.path.exists("checkpoints/loss_history.json"):
    with open("checkpoints/loss_history.json", "r") as f:
        history = json.load(f)

    epochs = history["epoch"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 1. Train Loss vs Val Loss
    axes[0].plot(epochs, history["train_loss"], 'b-o', lw=2, label="Train Loss")
    axes[0].plot(epochs, history["val_loss"], 'r-o', lw=2, label="Val Loss")
    axes[0].set_title("1. Динамика Train и Val Loss", fontsize=12)
    axes[0].set_xlabel("Эпоха")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # 2. Learning Rate
    axes[1].plot(epochs, history["lr"], 'g-o', lw=2)
    axes[1].set_title("2. График снижения Learning Rate (Cosine Annealing)", fontsize=12)
    axes[1].set_xlabel("Эпоха")
    axes[1].set_ylabel("LR")
    axes[1].grid(True, alpha=0.3)

    plt.suptitle("📈 КРИВЫЕ СХОДИМОСТИ RESTORMER", fontsize=14, y=1.03)
    plt.tight_layout()
    plt.savefig("checkpoints/loss_plot.png", dpi=300)
    plt.show()
    print("🎉 График сохранен в файл 'checkpoints/loss_plot.png'!")

# МОДУЛЬ 8: Визуализация 64 патчей тестового снимка

In [ ]:
# ==============================================================================
# МОДУЛЬ 8: Детальный визуальный аудит 64 патчей случайного снимка из TEST SET
# ==============================================================================
def visualize_all_patches_for_random_test_image(
    model,
    test_dataset,
    patch_size=512,
    overlap_ratio=0.2,
    max_patches_to_show=10,
    checkpoint_path="checkpoints/restormer_best_model.pth",
    device=device
):
    if os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print(f"✅ Успешно загружены лучшие веса из: '{checkpoint_path}'")
    model.eval()

    # Выбираем случайный снимок из ТЕСТОВОЙ выборки
    rand_idx = random.randint(0, len(test_dataset.common_names) - 1)
    test_noisy_path = test_dataset.noisy_paths[rand_idx]
    test_clean_path = test_dataset.clean_paths[rand_idx]
    file_name = test_dataset.common_names[rand_idx]

    print(f"🎯 Выбран случайный тестовый снимок: '{file_name}' (Индекс в тесте: {rand_idx})")

    # Нарезка полного снимка
    noisy_patches, coords, orig_shape = slice_image_to_patches(
        test_noisy_path, patch_size=patch_size, overlap_ratio=overlap_ratio
    )
    clean_patches, _, _ = slice_image_to_patches(
        test_clean_path, patch_size=patch_size, overlap_ratio=overlap_ratio
    )

    total_patches = len(noisy_patches)
    cleaned_patches = []
    batch_size = 8

    with torch.no_grad():
        for i in range(0, total_patches, batch_size):
            batch_n = noisy_patches[i:i+batch_size]

            # Нормализация и генерация Canny
            batch_2ch = []
            for p in batch_n:
                p_u8 = ((p - np.min(p)) / (np.max(p) - np.min(p) + 1e-8) * 255.0).astype(np.uint8)
                p_norm = p_u8.astype(np.float32) / 255.0
                canny_u8 = cv2.Canny(cv2.GaussianBlur(p_u8, (3, 3), 1.0), 65, 140)
                canny_norm = canny_u8.astype(np.float32) / 255.0

                # 2 канала: [Noisy, Canny]
                tensor_2ch = np.stack([p_norm, canny_norm], axis=0)
                batch_2ch.append(tensor_2ch)

            batch_tensor = torch.from_numpy(np.array(batch_2ch)).float().to(device)

            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                preds = model(batch_tensor)
            preds_np = preds.squeeze(1).cpu().numpy()

            for p in preds_np:
                cleaned_patches.append(p)

    num_to_plot = total_patches if max_patches_to_show is None else min(max_patches_to_show, total_patches)
    print(f"🖼️ Отрисовываем {num_to_plot} патчей в формате [Вход | Выход Restormer | Эталон]...\n")

    fig, axes = plt.subplots(num_to_plot, 3, figsize=(15, num_to_plot * 4))

    if num_to_plot == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(num_to_plot):
        n_p = noisy_patches[i]
        c_p = clean_patches[i]
        p_p = cleaned_patches[i]
        y, x = coords[i]

        n_disp = (n_p - np.min(n_p)) / (np.max(n_p) - np.min(n_p) + 1e-8)
        c_disp = (c_p - np.min(c_p)) / (np.max(c_p) - np.min(c_p) + 1e-8)
        p_disp = np.clip(p_p, 0.0, 1.0)

        axes[i, 0].imshow(n_disp, cmap='gray')
        axes[i, 0].set_title(f"Патч #{i} (y={y}, x={x})\n1. Зашумленный вход", fontsize=10)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(p_disp, cmap='gray')
        axes[i, 1].set_title(f"Патч #{i} (y={y}, x={x})\n2. Очищенный Restormer", fontsize=10, color='darkgreen')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(c_disp, cmap='gray')
        axes[i, 2].set_title(f"Патч #{i} (y={y}, x={x})\n3. Чистый эталон", fontsize=10)
        axes[i, 2].axis('off')

    plt.suptitle(f"Сравнение патчей для тестового снимка '{file_name}'", fontsize=15, y=1.001)
    plt.tight_layout()
    plt.show()

# Запуск визуализации
visualize_all_patches_for_random_test_image(
    model=model,
    test_dataset=test_dataset,
    patch_size=512,
    overlap_ratio=0.2,
    max_patches_to_show=10,
    checkpoint_path="checkpoints/restormer_best_model.pth",
    device=device
)

In [18]:
import torch
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"✅ VRAM сброшена! Свободно: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} ГБ")

✅ VRAM сброшена! Свободно: 31.7 ГБ


In [ ]:
model = GuidedRestormer().to(device)   # тот же класс, что использовался при обучении
model.load_state_dict(torch.load("checkpoints/restormer_best_model.pth", map_location=device))
model.eval()

Unknown instance spec: Please select VM configuration